In [1]:
import pandas as pd
import numpy as np
import scipy
import scipy.sparse as sp
import scipy.io as sio
import scipy.stats as stats
from tqdm.notebook import tqdm


import os
os.environ["R_HOME"] = f"{os.environ['CONDA_PREFIX']}\\Lib\\R"


from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from plotnine import *

import matplotlib.pyplot as plt 

import pickle

from joblib import Parallel, delayed

import sys
from pathlib import Path

# Get project root as parent of notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.methods.GWASH_funcs import *
from src.methods.GWASH_sim_funcs import *
from src.methods.ldsc_barebones import *

from src.simtools.sim_utils import *
from src.simtools.data_generation import data_generation as data_generation
from src.simtools.data_preprocessing import data_preprocessing as data_preprocessing
from src.simtools.do_analysis import do_analysis as do_analysis
from src.simtools.run_simulations import run_simulations as run_simulations
from src.simtools.data_generation import gen_ref_ldscores_panel as gen_ref_ldscores_panel
from src.simtools.visualization import visualize

from natsort import natsorted

import pickle

seed_num = 123
np.random.seed(seed_num)

os.chdir(PROJECT_ROOT)

In [2]:
n = 5000
m = 10000

seed_num = 123

X_properties = {'n': n,'m':m,'pm_causal':None,'fixed_m_causal': False,'sigma_s':0, 'rho1':0.995,'rho2':None,'Fst':0}
ref_X_properties = {'ref_n': n,'ref_pm_causal':None,'ref_fixed_m_causal': False,'ref_sigma_s':0, 'ref_rho1':0.995,'ref_rho2':None,'ref_Fst':0}
ld_mat_properties = {'realistic': True,'model_Fst_in_realistic':True,'prefix': 'D:/PhD/gwash_sim_test/to_github_restore/final_repo/1kg_p1_eur_ben/1kg_p1_eur_chr22','make_ref_ldscores':True}
method_properties = {'reml_tol': 1e-8,'reml_max_iters': 100}
stats_properties = {'scaleX': True,'scaley': True, 'nPCs':5,'regress_PC_out':False,'regress_X_on_PC': False,'regress_y_on_PC': False}
simul_properties = {'num_sims':np.nan,'h2_pop': 0.2}
debug_properties = {'calc_mu_hat_2_fast': True,'old':False,'track_progress':True}

list_of_dicts = [X_properties,ref_X_properties,ld_mat_properties,method_properties,stats_properties,simul_properties,debug_properties]
params  = combine_all_dicts(list_of_dicts)

to_run = dict()
to_run['demonstration'] = params

sim_key = 'demonstration'
locals().update(to_run[sim_key])
res_dict = dict()
res_dict_raw = dict()

#rhos = [0.995]
counter = -1
multithreading = True


counter += 1
Fsts = [0,0.0025,0.005,0.0075,0.01]
sigma_s = 0
if make_ref_ldscores:
    ref_data_gen = data_generation(n = ref_n,m= m,Fst = Fst,rho1 = ref_rho1,rho2 = ref_rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = ref_fixed_m_causal, old = old,realistic = realistic,prefix = prefix,use_gwash_m = False)
    ref_ldscores,ref_mu2_hat,ref_mu3_hat,X_ref = gen_ref_ldscores_panel(ref_data_gen,seed = seed_num,nPCs = nPCs, regress_X_on_PC = regress_X_on_PC, regress_y_on_PC = regress_y_on_PC)
    n_tilde = ref_data_gen.n
else:
    ref_data_gen = None
    ref_ldscores = None
    n_tilde = None

    ref_mu2_hat = None
    ref_mu3_hat = None
    X_ref = None

num_sims = 1
ldsc_h2s = []
ldsc_intercepts = []
ldsc_fixed_h2s = []

i = 0
np.random.seed(seed_num+i)

n_jobs = 1

my_data_gen = data_generation(n = n,m= m,Fst = Fst,rho1 = rho1,rho2 = rho2,sigma_s = sigma_s,h2_pop = h2_pop,pm_causal = pm_causal,fixed_m_causal = fixed_m_causal, old = old,model_Fst_in_realistic = model_Fst_in_realistic,realistic = realistic,prefix = prefix,ref_ldscores = ref_ldscores,ref_mu2_hat = ref_mu2_hat,ref_mu3_hat = ref_mu3_hat)

u2_res = np.zeros((m,num_sims))
ld_scores_res = np.zeros((m,num_sims))

X,b = my_data_gen.gen_X()
#b = np.zeros(m).reshape(-1,1)
y = my_data_gen.gen_y(X,b)

my_data_preprocessing = data_preprocessing(my_data_gen,nPCs = nPCs,regress_X_on_PC = regress_X_on_PC,regress_y_on_PC = regress_y_on_PC)

h2_samp_fve =  np.var(np.matmul(X,b),ddof=1)/np.var(y,ddof = 1)
if scaleX:
    X_tilde = my_data_preprocessing.manual_standardize_and_scale_inR(X)
else:
    X_tilde = X
if scaley:
    y_tilde = my_data_preprocessing.manual_standardize_and_scale_inR(y)
else:
    y_tilde = y


u2,s2,ldscores = my_data_gen.compute_u2_ldscores(X_tilde,y_tilde)

# u2_res[:,i] = u2.flatten()
# ld_scores_res[:,i] = ldscores.flatten()
GRM,GRM_id,pheno = my_data_gen.create_GCTA_inputs(X_tilde,y_tilde) # GCTA related inputs
input_GRM,input_pheno = my_data_gen.format_gcta_to_mats(GRM,pheno)

ldsc_reg_weights = ldscores
sims = do_analysis(my_data_gen,X_tilde,u2,s2,ldscores,ldsc_reg_weights,calc_mu_hat_2_fast = True,do_fast_GCTA = False)
old_GCTA = sims.do_GCTA(input_pheno,input_GRM,GRM_id,tol = 1e-8, iter_limit = 100,use_native_gcta = False,native_gcta_path = None)
sims = do_analysis(my_data_gen,X_tilde,u2,s2,ldscores,ldsc_reg_weights,calc_mu_hat_2_fast = True,do_fast_GCTA = True)
new_GCTA = sims.do_GCTA(input_pheno,input_GRM,GRM_id,tol = 1e-8, iter_limit = 100,use_native_gcta = False,native_gcta_path = None)

In [3]:
old_GCTA

{'VG': 0.18841837788047733,
 'Ve': 0.8220420489205725,
 'h2_est': 0.18646784464086208,
 'num_iters': 9,
 'time_elapsed': 59.30950470000971}

In [4]:
new_GCTA

{'VG': 0.1884183778804224,
 'Ve': 0.8220420489205764,
 'h2_est': 0.1864678446408171,
 'num_iters': 9,
 'time_elapsed': 13.884737399988808}

In [ ]:
"""
gcta_numba_helpers.py
Numba-accelerated helper functions for GCTA REML estimation (float64, Cholesky, module-level for JIT).
"""
import numpy as np
from numba import njit

@njit(cache=True)
def do_slogdet_cholesky(V):
    L = np.linalg.cholesky(V)
    return 2.0 * np.sum(np.log(np.diag(L)))
    
@njit(cache=True)
def compute_A_inv_w_solve(A):
    return np.linalg.solve(A,np.eye(A.shape[0]))

@njit(cache=True)
def calc_P_numba(V, X):
    n = V.shape[0]
    V_inv = compute_A_inv_w_solve(V)
    if X is None:
        X = np.ones((n, 1), dtype=np.float64)
    XtVinvX = X.T @ V_inv @ X
    XtVinvX_inv = compute_A_inv_w_solve(XtVinvX)
    rhs = (V_inv @ X) @ XtVinvX_inv @ X.T @ V_inv
    return V_inv - rhs

@njit(cache=True)
def make_AI_matrix_numba(y, P, A):
    tl = (y.T @ P @ A @ P @ A @ P @ y).item()
    tr = (y.T @ P @ A @ P @ P @ y).item()
    bl = (y.T @ P @ P @ A @ P @ y).item()
    br = (y.T @ P @ P @ P @ y).item()
    return 0.5 * np.array([[tl, tr], [bl, br]], dtype=np.float64)

@njit(cache=True)
def make_deriv_matrix_numba(y, P, A):
    top = (np.sum(P * A) - (y.T @ P @ A @ P @ y)).item()
    bot = (np.trace(P) - (y.T @ P @ P @ y)).item()
    return -0.5 * np.array([[top], [bot]], dtype=np.float64)

@njit(cache=True)
def loglik_numba(V, X, y):
    n = V.shape[0]
    V_inv = compute_A_inv_w_solve(V)
    if X is None:
        X = np.ones((n, 1), dtype=np.float64)
    middle = do_slogdet_cholesky(X.T @ V_inv @ X)
    left = do_slogdet_cholesky(V)
    P = calc_P_numba(V, X)
    right = y.T @ P @ y
    return (-0.5 * (left + middle + right)).item()

@njit(cache=True)
def parse_my_gcta_numba(params):
    VG = params[0, 0]
    Ve = params[1, 0]
    return VG, Ve

@njit(cache=True)
def make_mat_vec_prods(y,P,A):
    Py = P @ y
    APy = A @ Py
    PAPy = P @ APy
    PPy = P @ Py
    APAPy = A @ PAPy
    return Py,APy,PAPy,PPy,APAPy



@njit(cache=True)
def make_AI_matrix_prec(Py,APy,PAPy,PPy,APAPy):
    #tl = (y.T @ P @ A @ P @ A @ P @ y).item()
    tl = np.dot(Py.T,APAPy).item()
    #tr = (y.T @ P @ A @ P @ P @ y).item()
    tr = np.dot(APy.T,PPy).item()
    #bl = (y.T @ P @ P @ A @ P @ y).item()
    bl = np.dot(PAPy.T,Py).item()
    #br = (y.T @ P @ P @ P @ y).item()
    br = np.dot(Py.T,PPy).item()
    return 0.5 * np.array([[tl, tr], [bl, br]], dtype=np.float64)


@njit(cache=True)
def make_deriv_matrix_prec(P,A, Py,APy):
    #top = (np.sum(P * A) - (y.T @ PAPy)).item()
    top = (np.sum(P * A) - np.dot(Py.T,APy)).item()
    #bot = (np.trace(P) - (y.T @ PPy)).item()
    bot = (np.trace(P) - np.dot(Py.T,Py)).item()
    return -0.5 * np.array([[top], [bot]], dtype=np.float64)

@njit(cache=True)
def loglik_prec(V, X, y,Py):
    n = V.shape[0]
    V_inv = compute_A_inv_w_solve(V)
    if X is None:
        X = np.ones((n, 1), dtype=np.float64)
    middle = do_slogdet_cholesky(X.T @ V_inv @ X)
    left = do_slogdet_cholesky(V)
    P = calc_P_numba(V, X)
    #right = y.T @ P @ y
    right = np.dot(Py,y)
    return (-0.5 * (left + middle + right)).item()

def do_py_GCTA_fast(y, kinship, tol=1e-4, iter_limit=100):
    start = time.perf_counter()
    n = y.shape[0]
    y = y.astype(np.float64)
    I_n = np.eye(n, dtype=np.float64)
    kinship = kinship.astype(np.float64)

    # eigendecompose kinship once — Q and eigenvalues reused every iteration
    eigenvalues, Q = np.linalg.eigh(kinship)
    ones = np.ones((n, 1), dtype=np.float64)

    def make_P_from_eigen(sigma2_g, sigma2_e):
        d = sigma2_g * eigenvalues + sigma2_e   # O(n)
        V_inv = (Q / d) @ Q.T                   # O(n²)
        V_inv_1 = V_inv @ ones
        denom = (ones.T @ V_inv_1).item()
        P = V_inv - (V_inv_1 @ V_inv_1.T) / denom
        return P, d, V_inv_1, denom

    def loglik_from_eigen(d, V_inv_1, denom, P, y):
        log_det_V = np.sum(np.log(d))           # O(n)
        log_det_XVX = np.log(denom)
        yPy = (y.T @ P @ y).item()
        return -0.5 * (log_det_V + log_det_XVX + yPy)

    y_temp = y - np.mean(y)
    y_temp_SSq = ((y_temp ** 2).sum()) / (n - 1)
    sigma2_g, sigma2_e = y_temp_SSq / 2, y_temp_SSq / 2

    P, d, V_inv_1, denom = make_P_from_eigen(sigma2_g, sigma2_e)
    LL0 = loglik_from_eigen(d, V_inv_1, denom, P, y)

    # initial EM step (unchanged)
    rhs_g = sigma2_g * n - sigma2_g ** 2 * np.sum(P * kinship)
    sigma2_g = ((sigma2_g ** 2 * y.T @ P @ kinship @ P @ y + rhs_g) / n).item()
    rhs_e = np.trace(sigma2_e * I_n - sigma2_e ** 2 * P)
    sigma2_e = ((sigma2_e ** 2 * y.T @ P @ P @ y + rhs_e) / n).item()

    stop = False
    params = np.array([sigma2_g, sigma2_e], dtype=np.float64).reshape(-1, 1)
    counter = 0

    while not stop:
        sigma2_g = params[0, 0]
        sigma2_e = params[1, 0]
        if sigma2_g < 0:
            sigma2_g = y_temp_SSq * (10 ** -6)
        if sigma2_e < 0:
            sigma2_e = y_temp_SSq * (10 ** -6)
        params[0, 0] = sigma2_g
        params[1, 0] = sigma2_e

        P, d, V_inv_1, denom = make_P_from_eigen(sigma2_g, sigma2_e)
        LL = loglik_from_eigen(d, V_inv_1, denom, P, y)

        if abs(LL - LL0) < tol:
            stop = True
        else:
            LL0 = LL
            Py,APy,PAPy,PPy,APAPy = make_mat_vec_prods(y,P,kinship)
            #AI_mat = make_AI_matrix_numba(y, P, kinship)
            AI_mat = make_AI_matrix_prec(Py,APy,PAPy,PPy,APAPy)
            #deriv_mat = make_deriv_matrix_numba(y, P, kinship)
            deriv_mat = make_deriv_matrix_prec(P,kinship, Py,APy)
            params = params + (compute_A_inv_w_solve(AI_mat) @ deriv_mat)
            counter += 1
        if counter > iter_limit:
            logging.info('GCTA exceeded iteration limit')
            stop = True

    VG, Ve = parse_my_gcta_numba(params)
    VP = VG + Ve
    h2_est = VG / VP
    res_dict = dict()
    end = time.perf_counter()
    res_dict['VG'] = VG
    res_dict['Ve'] = Ve
    res_dict['h2_est'] = h2_est
    res_dict['num_iters'] = counter
    res_dict['time_elapsed'] = end - start
    return res_dict



In [ ]:
new_GCTA = do_py_GCTA_fast(input_pheno,input_GRM, tol=1e-8, iter_limit=100)

In [ ]:
old_GCTA

In [ ]:
new_GCTA

In [ ]:
new_GCTA['h2_est'] - old_GCTA['h2_est']

In [ ]:
import tracemalloc

tracemalloc.start()
old_GCTA = sims.do_GCTA(input_pheno,input_GRM,GRM_id,tol = 1e-8, iter_limit = 100,use_native_gcta = False,native_gcta_path = None)

current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()



In [ ]:
print(f"Peak memory: {peak / 1e6:.1f} MB") #1.4 GB

In [ ]:
tracemalloc.start()

new_GCTA = do_py_GCTA_fast(input_pheno,input_GRM, tol=1e-8, iter_limit=100)

current_new, peak_new = tracemalloc.get_traced_memory()

tracemalloc.stop()


print(f"Peak memory: {peak_new / 1e6:.1f} MB") #1.4 GB